# Lab 01 — Primeiro contato com dados

**Onde roda:** 🟢 Browser (JupyterLite) ou 🐳 Bancada Docker. Execute célula a célula (Shift+Enter).

```{tip}
**▶ Rodar no navegador (JupyterLite):** <a href="../../lite/lab/index.html?path=lab-01-primeiro-contato-dados.ipynb" target="_blank" rel="noopener">abrir este notebook interativo</a>. A célula de setup instala o `duckdb` sob demanda (alguns segundos na primeira vez).
```

Objetivo: sentir na prática a diferença entre olhar dados com **pandas** (orientado a linhas, em Python) e consultá-los com **SQL/DuckDB** (orientado a análise). Este é o gesto mais básico e mais repetido da Engenharia de Dados: **carregar e inspecionar**.

In [ ]:
# Setup: garante o duckdb.
# No navegador (JupyterLite) ele é instalado sob demanda; na bancada Docker já vem instalado.
try:
    import duckdb
except ModuleNotFoundError:
    import piplite
    await piplite.install("duckdb")
    import duckdb
print("duckdb", duckdb.__version__, "pronto")

## 1. Criando um dataset de exemplo
Para o lab ser autocontido (roda no navegador, sem baixar nada), criamos um pequeno conjunto de vendas em memória.

In [ ]:
import pandas as pd

vendas = pd.DataFrame({
    'pedido': [1, 2, 3, 4, 5, 6],
    'estado': ['SP', 'SP', 'RJ', 'MG', 'RJ', 'SP'],
    'categoria': ['livros', 'eletronicos', 'livros', 'eletronicos', 'livros', 'eletronicos'],
    'valor': [50.0, 1200.0, 30.0, 800.0, 45.0, 1500.0],
})
vendas

## 2. Explorando com pandas
pandas é ótimo para inspecionar e manipular em Python.

In [ ]:
print('Linhas x colunas:', vendas.shape)
print('\nReceita total por estado:')
print(vendas.groupby('estado')['valor'].sum().sort_values(ascending=False))

## 3. A mesma pergunta com SQL (DuckDB)
DuckDB é um banco analítico (OLAP) que roda embarcado — inclusive no navegador. A mesma agregação, em SQL:

In [ ]:
import duckdb

duckdb.query('''
    SELECT estado, SUM(valor) AS receita
    FROM vendas
    GROUP BY estado
    ORDER BY receita DESC
''').to_df()

> **Reflexão:** o mesmo resultado, dois caminhos. pandas vive no processo Python; SQL/DuckDB expressa a análise de forma declarativa e escala melhor para dados grandes e colunares. Ao longo do curso você usará os dois — e entenderá quando cada um brilha.

## 4. Sua vez (mini-desafio)
Escreva uma consulta que traga a **receita por categoria**, da maior para a menor. Depois rode a célula de verificação.

In [ ]:
# Complete a query:
resposta = duckdb.query('''
    SELECT categoria, SUM(valor) AS receita
    FROM vendas
    GROUP BY categoria
    ORDER BY receita DESC
''').to_df()
resposta

In [ ]:
# Verificacao (roda no navegador)
def verificar(df):
    try:
        assert list(df.columns) == ['categoria', 'receita'], 'As colunas devem ser categoria e receita.'
        top = df.iloc[0]
        assert top['categoria'] == 'eletronicos', 'A categoria de maior receita deveria ser eletronicos.'
        assert abs(float(top['receita']) - 3500.0) < 1e-6, 'A receita de eletronicos deveria ser 3500.'
        print('\u2705 Correto! Voce agregou e ordenou certo.')
    except AssertionError as e:
        print('\u274c', e)

verificar(resposta)